# Zava Retail Agent: SFT, RFT, deployment, and comparison

Preserve the original GPT models, data, training recipes, and graders. Run the local preparation cells to validate inputs and inspect methods. Subsequent cells train, verify existing deployments, and run inference comparisons; training and inference can incur charges.

Actual results may vary by model version, data, configuration, region availability, and service conditions.

## 1. Customer configuration

Install the repository shared dependencies before running. Copy this cookbook's `.env.template` to a module-local `.env` in this cookbook directory (not the repository/shared directory). Supply your own project endpoints and unique deployment names; no personal resource defaults are provided. Executing the upload/training and inference cells performs billable operations. Execute only the stages you intend to run. Deployment readiness checks are read-only; the notebook never creates or replaces deployments.

RFT training retains mutating tools. Use an approved HTTPS tool server backed by an isolated disposable database, not production, and acknowledge it with `FOUNDRY_TOOL_SERVER_IS_SANDBOX=true`. Comparison blocks mutating/unknown tool calls independently of this acknowledgement.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import importlib.util
import io
import json
import math
import os
import re
import sys
import time
import uuid
from pathlib import Path
from urllib.parse import quote, urlparse
from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'manifest.json').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'manifest.json').is_file():
    raise RuntimeError('Run from the cookbook directory or notebooks/.')
sys.path.insert(0, str(ROOT.parents[1]))
from shared.retry import retry_call
from shared.config import load_foundry_config
from shared.foundry_operations import wait_for_file, monitor_fine_tuning_job

load_dotenv(ROOT / '.env', override=False)

def setting(name, default=''):
    return os.getenv(name, '').strip() or default

def env_flag(name):
    value = setting(name, 'false').casefold()
    if value not in {'true', 'false'}:
        raise ValueError(f'{name} must be true or false (blank means false).')
    return value == 'true'

def required(name):
    value = setting(name)
    if not value:
        raise ValueError(f'Set {name} explicitly before running this stage.')
    return value

def positive_integer(name, default=None):
    value = setting(name, str(default) if default is not None else '')
    if not value.isdecimal() or int(value) < 1:
        raise ValueError(f'{name} must be an explicit positive integer.')
    return int(value)

def https_url(value, label):
    parsed = urlparse(value)
    if parsed.scheme != 'https' or not parsed.hostname or parsed.username or parsed.password or parsed.query or parsed.fragment:
        raise ValueError(f'{label} must be an HTTPS URL without credentials, query, or fragment.')
    return value.rstrip('/')

def project_endpoint(name):
    return load_foundry_config(environ={'FOUNDRY_PROJECT_ENDPOINT': required(name)}).project_endpoint

SFT_MODEL = 'gpt-4.1-mini-2025-04-14'
RFT_MODEL = 'o4-mini-2025-04-16'
TRAINING_TYPE = 'globalStandard'
SFT_ENDPOINT = setting('FOUNDRY_SFT_PROJECT_ENDPOINT')
RFT_ENDPOINT = setting('FOUNDRY_RFT_PROJECT_ENDPOINT')
TOOL_SERVER_URL = setting('FOUNDRY_TOOLS_SERVER_URL').rstrip('/')
POLL_SECONDS = positive_integer('FOUNDRY_POLL_INTERVAL_SECONDS', 60)
WORKERS = positive_integer('FOUNDRY_COMPARISON_WORKERS', 1)
DATA, CONFIGS, OUTPUTS = ROOT / 'data', ROOT / 'configs', ROOT / 'outputs'
STATE_PATH = OUTPUTS / 'notebook-state.json'


## 2. Validate exact original inputs

Every preserved input is checked against its LF-canonical byte count, SHA-256 hash, and JSONL record count. CRLF is normalized only for lineage checks; source files and upload payloads remain unchanged.

In [ ]:
manifest = json.loads((DATA / 'manifest.json').read_text(encoding='utf-8'))
validated_inputs = {}
for entry in manifest['files']:
    path = ROOT / entry['path']
    canonical = path.read_bytes().replace(b'\r\n', b'\n')
    digest = hashlib.sha256(canonical).hexdigest()
    if len(canonical) != entry['bytes'] or digest != entry['sha256']:
        raise RuntimeError(f"Lineage mismatch for {entry['path']}")
    if path.suffix == '.jsonl':
        rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
        if len(rows) != entry['records']:
            raise RuntimeError(f"Row-count mismatch for {entry['path']}")
    validated_inputs[entry['path']] = digest
print('Verified original training and comparison inputs.')

## 3. Preserve the original training recipes

SFT: batch size 1, learning-rate multiplier 2, three epochs. RFT: original o3-mini policy grader and remote tools, one epoch. The placeholder tool URL remains local until training is enabled.

In [ ]:
policy_grader = json.loads((CONFIGS / 'rft-policy-grader.json').read_text(encoding='utf-8'))
tools = json.loads((CONFIGS / 'rft-tools.json').read_text(encoding='utf-8'))
if TOOL_SERVER_URL:
    for tool in tools:
        tool['server_url'] = TOOL_SERVER_URL
sft_method = {'type': 'supervised', 'supervised': {'hyperparameters': {
    'batch_size': 1, 'learning_rate_multiplier': 2, 'n_epochs': 3,
}}}
rft_method = {'type': 'reinforcement', 'reinforcement': {
    'hyperparameters': {'eval_interval': 1, 'eval_samples': 2, 'compute_multiplier': 1,
        'reasoning_effort': 'medium', 'n_epochs': 1, 'batch_size': 10, 'learning_rate_multiplier': 1},
    'grader': policy_grader, 'tools': tools,
}}
print({'sft_model': SFT_MODEL, 'rft_model': RFT_MODEL, 'grader': policy_grader['model']})

## 4. Authenticate only for enabled stages

Use `az login` before enabling a service stage. All model operations use the OpenAI clients returned by `AIProjectClient.get_openai_client()`. Automatic SDK retries are disabled for writes; shared bounded retries are used only for reads/inference.

In [ ]:
resources_to_close = []
SFT_ENDPOINT = project_endpoint('FOUNDRY_SFT_PROJECT_ENDPOINT')
RFT_ENDPOINT = project_endpoint('FOUNDRY_RFT_PROJECT_ENDPOINT')
TOOL_SERVER_URL = https_url(required('FOUNDRY_TOOLS_SERVER_URL'), 'Tool server')
if not env_flag('FOUNDRY_TOOL_SERVER_IS_SANDBOX'):
    raise ValueError('RFT requires an explicitly acknowledged isolated sandbox tool server.')
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
credential = DefaultAzureCredential()
resources_to_close.append(credential)
for label, endpoint in [('sft', SFT_ENDPOINT), ('rft', RFT_ENDPOINT)]:
    project = AIProjectClient(endpoint=endpoint, credential=credential, allow_preview=True)
    resources_to_close.append(project)
    client = project.get_openai_client(max_retries=0)
    resources_to_close.append(client)
    if label == 'sft':
        sft_client = client
    else:
        rft_client = client


## 5. Optional upload, submission, and monitoring

State is bound to customer endpoints, exact inputs, models, and methods. Keep it to resume. A pending upload/submission after an interrupted request fails closed: reconcile the recorded unique request with your project before editing state. Never delete state merely to retry an uncertain paid submission. The visible SDK submission preserves `trainingType=globalStandard`, which the shared job helper does not accept; SDK write retries are disabled and pending state is saved before the POST. Monitoring uses the shared helper, has a 24-hour timeout, and does not cancel jobs.

In [ ]:
def save_state():
    OUTPUTS.mkdir(exist_ok=True)
    temporary = STATE_PATH.with_suffix('.tmp')
    temporary.write_text(json.dumps(state, indent=2, sort_keys=True), encoding='utf-8')
    temporary.replace(STATE_PATH)

context = {'endpoints': [SFT_ENDPOINT, RFT_ENDPOINT], 'inputs': validated_inputs,
    'models': [SFT_MODEL, RFT_MODEL], 'training_type': TRAINING_TYPE,
    'methods': [sft_method, rft_method]}
state = None
if STATE_PATH.is_file():
    state = json.loads(STATE_PATH.read_text(encoding='utf-8'))
    if state.get('context') != context:
        raise RuntimeError('Runtime state belongs to different inputs/configuration. Reconcile it explicitly.')
else:
    state = {'context': context, 'run_token': uuid.uuid4().hex[:16], 'files': {}, 'jobs': {}}
    save_state()

def upload_once(client, label, path):
    record = state['files'].get(label)
    if record is None:
        state['files'][label] = {'pending': True, 'filename': f"retail-{state['run_token']}-{path.name}"}
        save_state()
        with path.open('rb') as stream:
            uploaded = client.files.create(file=(state['files'][label]['filename'], stream, 'application/jsonl'), purpose='fine-tune')
        record = state['files'][label] = {'id': uploaded.id}
        save_state()
    if not record.get('id'):
        raise RuntimeError(f'Uncertain upload for {label}; reconcile the recorded filename before retrying.')
    wait_for_file(client, record['id'], poll_interval_seconds=10, timeout_seconds=600)
    return record['id']

def submit_once(client, label, model, method):
    record = state['jobs'].get(label)
    if record is None:
        suffix = f"retail-{label}-{state['run_token']}"
        state['jobs'][label] = {'pending': True, 'suffix': suffix}
        save_state()
        job = client.fine_tuning.jobs.create(model=model,
            training_file=state['files'][f'{label}_train']['id'],
            validation_file=state['files'][f'{label}_validation']['id'],
            suffix=suffix, method=method, extra_body={'trainingType': TRAINING_TYPE})
        record = state['jobs'][label] = {'id': job.id, 'suffix': suffix}
        save_state()
    if not record.get('id'):
        raise RuntimeError(f'Uncertain submission for {label}; reconcile the recorded suffix before retrying.')
    return record['id']



### Upload, train, and monitor

Execute this cell to upload inputs and submit or resume both paid jobs. For comparisons using saved successful jobs, skip this cell and the metric-download cell, then run deployment readiness and comparison cells.


In [ ]:
for label, client in [('sft', sft_client), ('rft', rft_client)]:
    for split in ['train', 'validation']:
        upload_once(client, f'{label}_{split}', DATA / f'{label}-{split}.jsonl')
submit_once(sft_client, 'sft', SFT_MODEL, sft_method)
submit_once(rft_client, 'rft', RFT_MODEL, rft_method)
for label, client in [('sft', sft_client), ('rft', rft_client)]:
    job = monitor_fine_tuning_job(client, state['jobs'][label]['id'], poll_interval_seconds=POLL_SECONDS)
    state['jobs'][label].update(status=job.status, fine_tuned_model=job.fine_tuned_model, result_files=job.result_files or [])
    save_state()


## 6. Optional service metrics

Download only service-produced CSVs; absence of expected metrics is an error, not an invented curve.

In [ ]:
def download_result(client, label):
    record = state['jobs'][label]
    if not record['result_files']:
        raise RuntimeError(f'{label} succeeded without a service result file.')
    content = retry_call(lambda: client.files.content(record['result_files'][0]).read(), operation_name='download metrics')
    (OUTPUTS / f'{label}-results.csv').write_bytes(content)
    return list(csv.DictReader(io.StringIO(content.decode('utf-8-sig'))))

def metric_curve(rows, field):
    points = [(int(row['step']), float(row[field])) for row in rows if row.get(field)]
    if not points or not all(math.isfinite(value) for _, value in points):
        raise RuntimeError(f'Missing or non-finite service metric: {field}')
    return points

import matplotlib.pyplot as plt
sft_rows = download_result(sft_client, 'sft')
rft_rows = download_result(rft_client, 'rft')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(*zip(*metric_curve(sft_rows, 'valid_loss')), marker='o')
axes[0].set(title='SFT service validation loss', xlabel='Step', ylabel='Loss')
for field, label in [('train_mean_reward', 'Training'), ('full_valid_mean_reward', 'Validation')]:
    axes[1].plot(*zip(*metric_curve(rft_rows, field)), marker='o', label=label)
axes[1].set(title='RFT service reward', xlabel='Step', ylabel='Reward')
axes[1].legend()
plt.tight_layout()


## 7. Provision through your approved process, then verify readiness

After successful training, provision GlobalStandard deployments through your approved Foundry portal or management workflow using the saved fine-tuned artifacts in `outputs/notebook-state.json`. Choose four distinct unique deployment names and appropriate capacity in that workflow. This notebook only discovers and verifies existing deployments: it never creates, updates, or deletes resources, and makes no claim about ARM conditional-create support. Missing resources, authorization errors, and model/version/format/SKU mismatches fail explicitly; provisioning polling is bounded.

SFT base and fine-tuned deployments use the SFT account; RFT base and fine-tuned deployments use the RFT account. Configure an existing o3-mini grader separately. To compare preprovisioned deployments, retain successful training state/configuration, run the saved-state loading and deployment-readiness cells, and then run the comparison cells. Skip the upload/training cells when the recorded jobs have already completed. No creation guard flag, capacity setting, or UUID suffix is required.

In [ ]:
def deployment_specifications():
    specs = []
    names = [required(name) for name in ['FOUNDRY_SFT_BASE_DEPLOYMENT', 'FOUNDRY_SFT_DEPLOYMENT', 'FOUNDRY_RFT_BASE_DEPLOYMENT', 'FOUNDRY_RFT_DEPLOYMENT']]
    if len(set(names)) != 4 or any(not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,63}', name) for name in names):
        raise ValueError('Choose four distinct valid unique deployment names.')
    for index, (label, endpoint, model, version) in enumerate([
        ('sft', SFT_ENDPOINT, 'gpt-4.1-mini', '2025-04-14'),
        ('sft', SFT_ENDPOINT, state['jobs'].get('sft', {}).get('fine_tuned_model'), '1'),
        ('rft', RFT_ENDPOINT, 'o4-mini', '2025-04-16'),
        ('rft', RFT_ENDPOINT, state['jobs'].get('rft', {}).get('fine_tuned_model'), '1'),
    ]):
        if not model or state['jobs'].get(label, {}).get('status') != 'succeeded':
            raise RuntimeError(f'{label} requires a saved successful model artifact.')
        account = required(f'FOUNDRY_{label.upper()}_ACCOUNT_NAME')
        if urlparse(endpoint).hostname != f'{account}.services.ai.azure.com':
            raise ValueError(f'{label} account must match the configured Foundry endpoint.')
        specs.append((account, names[index], model, version, label))
    return specs

def arm_url(account, name):
    subscription = quote(required('FOUNDRY_SUBSCRIPTION_ID'), safe='')
    group = quote(required('FOUNDRY_RESOURCE_GROUP'), safe='')
    return f'https://management.azure.com/subscriptions/{subscription}/resourceGroups/{group}/providers/Microsoft.CognitiveServices/accounts/{quote(account, safe="")}/deployments/{quote(name, safe="")}?api-version=2024-10-01'

def arm_headers():
    return {'Authorization': f"Bearer {credential.get_token('https://management.azure.com/.default').token}", 'Content-Type': 'application/json'}

def verify_deployment(resource, model, version):
    actual = resource.get('properties', {}).get('model', {})
    expected = {'format': 'OpenAI', 'name': model, 'version': version}
    if any(actual.get(key) != value for key, value in expected.items()) or resource.get('sku', {}).get('name') != 'GlobalStandard':
        raise RuntimeError('Deployment name collision: model/version/format/SKU differs. Choose a new unique name; no resource was changed.')

def ensure_deployment(account, name, model, version):
    import requests
    url = arm_url(account, name)
    current = requests.get(url, headers=arm_headers(), timeout=60)
    if current.status_code == 404:
        raise RuntimeError(f'Missing deployment {name}; provision it through your approved process, then rerun discovery.')
    current.raise_for_status()
    verify_deployment(current.json(), model, version)
    deadline = time.monotonic() + 600
    while time.monotonic() < deadline:
        current = requests.get(url, headers=arm_headers(), timeout=60)
        current.raise_for_status()
        resource = current.json()
        verify_deployment(resource, model, version)
        status = resource['properties'].get('provisioningState')
        if status == 'Succeeded':
            return resource
        if status in {'Failed', 'Canceled', 'Cancelled', 'Deleting'}:
            raise RuntimeError(f'Deployment {name} ended in {status}. No cleanup operation is attempted.')
        time.sleep(15)
    raise TimeoutError(f'Deployment {name} did not become ready within 600 seconds.')

deployment_evidence = {}
specs = deployment_specifications()
required('FOUNDRY_SUBSCRIPTION_ID')
required('FOUNDRY_RESOURCE_GROUP')
for account, name, model, version, label in specs:
    deployment_evidence[name] = ensure_deployment(account, name, model, version)
(OUTPUTS / 'deployment-state.json').write_text(json.dumps(deployment_evidence, indent=2), encoding='utf-8')


## 8. Optional SFT invocation and base-versus-fine-tuned comparison

Both deployments receive all 280 original held-out next-tool decisions and identical generation settings. Each row must expect exactly one call. Only exactly one predicted call is graded, using the unchanged original grader; missing or extra predictions score zero. This prevents the original multi-call grader from double-crediting a row. Report single-call grader reward separately from exact/name accuracy. SFT only predicts calls; it never executes them.

In [ ]:
grader_spec = importlib.util.spec_from_file_location('retail_next_tool_grader', ROOT / 'graders' / 'next-tool-call-grader.py')
next_tool_grader = importlib.util.module_from_spec(grader_spec)
grader_spec.loader.exec_module(next_tool_grader)

def score_next_tool_row(calls, item):
    expected = item['expected_output']['tool_calls']
    if len(expected) != 1:
        raise ValueError('Original next-tool comparison rows must expect exactly one call.')
    if len(calls) != 1:
        return {'reward': 0.0, 'exact': False, 'tool_name_match': False}
    return {'reward': next_tool_grader.grade({'output_tools': calls}, item),
        'exact': next_tool_grader._sim(calls[0], expected[0]) == 1.0,
        'tool_name_match': calls[0]['function']['name'] == expected[0]['function']['name']}

from azure.ai.projects.models import ModelDeployment
GRADER_ENDPOINT = project_endpoint('FOUNDRY_GRADER_PROJECT_ENDPOINT')
GRADER_DEPLOYMENT = required('FOUNDRY_RFT_GRADER_DEPLOYMENT')
grader_project = AIProjectClient(endpoint=GRADER_ENDPOINT, credential=credential, allow_preview=True)
resources_to_close.append(grader_project)
grader_resource = grader_project.deployments.get(GRADER_DEPLOYMENT)
if not isinstance(grader_resource, ModelDeployment) or grader_resource.model_name != 'o3-mini':
    raise RuntimeError('Configured grader deployment must serve the original o3-mini model.')
grader_client = grader_project.get_openai_client(max_retries=0)
resources_to_close.append(grader_client)

def score_sft_deployment(deployment, rows):
    from concurrent.futures import ThreadPoolExecutor
    def score_one(row):
        response = retry_call(lambda: sft_client.chat.completions.create(model=deployment,
            messages=row['item']['messages'], tools=row['item']['tools'], tool_choice='required',
            temperature=0, max_completion_tokens=1024), operation_name='SFT comparison')
        calls = [call.model_dump(exclude_none=True) for call in response.choices[0].message.tool_calls or []]
        return score_next_tool_row(calls, row['item'])
    if not rows:
        raise ValueError('Comparison rows must not be empty.')
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        results = list(pool.map(score_one, rows))
    return {'total': len(results), 'mean_single_call_grader_reward': sum(item['reward'] for item in results) / len(results),
        'exact_rate': sum(item['exact'] for item in results) / len(results),
        'tool_name_match_rate': sum(item['tool_name_match'] for item in results) / len(results)}

sft_comparison = None
sft_eval_rows = [json.loads(line) for line in (DATA / 'next-tool-evaluation.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
sft_comparison = {label: score_sft_deployment(required(name), sft_eval_rows) for label, name in [
    ('base', 'FOUNDRY_SFT_BASE_DEPLOYMENT'), ('fine_tuned', 'FOUNDRY_SFT_DEPLOYMENT')]}
print(sft_comparison)


## 9. Optional read-only RFT invocation and policy comparison

Exclude scenario 7 on both sides and reject any remaining reference sequence containing unknown/mutating tools. A runtime allowlist blocks every model-generated mutating/unknown call before remote requests, even in otherwise read-only scenarios. Both models use the RFT client/account. Grade final answers with the original o3-mini policy prompt and pass threshold; malformed or out-of-range scores fail explicitly.

In [ ]:
READ_ONLY_TOOLS = frozenset({'calculate', 'find_user_id_by_email', 'find_user_id_by_name_zip',
    'get_order_details', 'get_product_details', 'get_user_details', 'list_all_product_types',
    'list_user_orders', 'policy_verify_return'})

def execute_remote_tool(call):
    if call.function.name not in READ_ONLY_TOOLS:
        raise RuntimeError(f'Blocked mutating or unknown tool: {call.function.name}')
    if not env_flag('FOUNDRY_TOOL_SERVER_IS_SANDBOX'):
        raise RuntimeError('Remote tools require an acknowledged isolated sandbox.')
    import requests
    payload = {'type': 'function_call', 'id': call.id, 'call_id': call.id,
        'name': call.function.name, 'arguments': call.function.arguments, 'trace_id': f'retail-{uuid.uuid4().hex}'}
    def request():
        response = requests.post(TOOL_SERVER_URL, json=payload, timeout=60, allow_redirects=False)
        response.raise_for_status()
        if response.is_redirect:
            raise RuntimeError('Tool server redirects are not permitted.')
        body = response.json()
        if 'output' not in body:
            raise RuntimeError('Tool server did not return an output field.')
        output = body['output']
        return output if isinstance(output, str) else json.dumps(output)
    return retry_call(request, operation_name='read-only retail tool')

def invoke_rft(client, deployment, row):
    messages, called_tools = list(row['messages']), []
    for _ in range(12):
        response = retry_call(lambda: client.chat.completions.create(model=deployment, messages=messages,
            tools=row['tools'], tool_choice='auto', max_completion_tokens=4096, reasoning_effort='medium'), operation_name='RFT invocation')
        message = response.choices[0].message
        if not message.tool_calls:
            if not message.content:
                raise RuntimeError('RFT returned no final answer or tool calls.')
            return message.content, called_tools
        if any(call.function.name not in READ_ONLY_TOOLS for call in message.tool_calls):
            raise RuntimeError('Blocked response containing mutating or unknown tools before executing any calls.')
        messages.append({'role': 'assistant', 'content': message.content,
            'tool_calls': [call.model_dump(exclude_none=True) for call in message.tool_calls]})
        for call in message.tool_calls:
            called_tools.append(call.function.name)
            messages.append({'role': 'tool', 'tool_call_id': call.id, 'content': execute_remote_tool(call)})
    raise RuntimeError('RFT invocation exceeded 12 model turns.')

def parse_policy_score(content):
    text = (content or '').strip()
    if not re.fullmatch(r'(?:0(?:\.\d+)?|1(?:\.0+)?)', text):
        raise RuntimeError('Policy grader must return only a numeric score in [0, 1].')
    return float(text)

def policy_score(row, output):
    replacements = {'{{item.reference_policy_args}}': json.dumps(row['reference_policy_args']),
        '{{item.reference_policy_outcome}}': json.dumps(row['reference_policy_outcome']), '{{sample.output_text}}': output}
    messages = [{'role': item['role'], 'content': re.sub(r'\{\{(?:item\.(?:reference_policy_args|reference_policy_outcome)|sample\.output_text)\}\}',
        lambda match: replacements[match.group(0)], item['content'])} for item in policy_grader['input']]
    response = retry_call(lambda: grader_client.chat.completions.create(model=GRADER_DEPLOYMENT, messages=messages,
        max_completion_tokens=4096, reasoning_effort='low'), operation_name='policy grading')
    return parse_policy_score(response.choices[0].message.content)

def select_read_only_scenarios(rows):
    selected = [(index, row) for index, row in enumerate(rows) if index != 7]
    for index, row in selected:
        if any(name not in READ_ONLY_TOOLS for name in row['reference_tool_calls']):
            raise RuntimeError(f'Scenario {index} has a mutating or unknown reference call.')
    if len(selected) != 9:
        raise RuntimeError('Expected the original nine read-only comparison scenarios.')
    return selected

def score_rft_deployment(client, deployment, rows):
    results = []
    for index, row in rows:
        output, called_tools = invoke_rft(client, deployment, row)
        results.append({'index': index, 'score': policy_score(row, output), 'called_tools': called_tools,
            'reference_tools': row['reference_tool_calls']})
    return {'mean_policy_reward': sum(item['score'] for item in results) / len(results),
        'pass_count': sum(item['score'] >= policy_grader['pass_threshold'] for item in results),
        'tool_sequence_exact': sum(item['called_tools'] == item['reference_tools'] for item in results),
        'total': len(results), 'results': results}

rft_rows = [json.loads(line) for line in (DATA / 'rft-validation.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
rft_selected = select_read_only_scenarios(rft_rows)
rft_comparison = None
rft_comparison = {label: score_rft_deployment(rft_client, required(name), rft_selected) for label, name in [
    ('base', 'FOUNDRY_RFT_BASE_DEPLOYMENT'), ('fine_tuned', 'FOUNDRY_RFT_DEPLOYMENT')]}
(OUTPUTS / 'comparison-state.json').write_text(json.dumps({'sft': sft_comparison, 'rft': rft_comparison}, indent=2), encoding='utf-8')
print(rft_comparison)


## 10. Interpret your own results

Training loss/reward are service telemetry, not held-out customer quality. Compare the same rows, generation settings, and original graders on each side. Fine-tuning is not guaranteed to improve reward; retain regressions in your conclusions. No customer endpoints, job IDs, historical run outputs, or deployment evidence are shipped.

The final cell closes created clients. If a service cell fails, run it before restarting the kernel. No resources are deleted or canceled.

In [ ]:
for resource in reversed(resources_to_close):
    resource.close()
resources_to_close.clear()